# 02a — Fine-tune with LoRA/PEFT + TRL `SFTTrainer`

**Purpose:** The most portable fine-tuning path. Runs parameter-efficient (LoRA) supervised fine-tuning on the ChatML dataset from `01` using Hugging Face `transformers` + `peft` + `trl` on a **single GPU (or multi-GPU) cluster**. Works on any GPU cluster with no managed services.

Sibling paths: `02b` (Ray Train, distributed) and `02c` (Databricks Mosaic AI, managed). All three consume the same dataset from `01` and share the same `BASE_MODEL` constant, so results are comparable.

> **Cluster:** single-node GPU (e.g. 1× A10 / A100). The default 1B model fits comfortably; larger models need more VRAM.

In [ ]:
%pip install -q -U transformers peft trl datasets accelerate bitsandbytes mlflow
dbutils.library.restartPython()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"   # swap for any HF causal LM (Qwen2.5-7B-Instruct, etc.)

CATALOG = "main"
SCHEMA  = "otel_finetuning"
VOLUME  = f"/Volumes/{CATALOG}/{SCHEMA}/finetune"
TRAIN_PATH = f"{VOLUME}/train.jsonl"
EVAL_PATH  = f"{VOLUME}/eval.jsonl"
OUTPUT_DIR = f"{VOLUME}/lora_adapter"

# LoRA
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training
EPOCHS        = 2
BATCH_SIZE    = 4
GRAD_ACCUM    = 4
LR            = 2e-4
MAX_SEQ_LEN   = 1024

MLFLOW_EXPERIMENT = "/Shared/otel-finetuning"

## Load the ChatML dataset

In [ ]:
from datasets import load_dataset

# TRL applies the model's chat template to the `messages` field automatically.
train_ds = load_dataset("json", data_files=TRAIN_PATH, split="train")
eval_ds  = load_dataset("json", data_files=EVAL_PATH,  split="train")
print(train_ds, eval_ds)
print(train_ds[0]["messages"])

## Load base model + tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

## Configure LoRA + `SFTTrainer` and train

In [ ]:
import mlflow
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES, task_type="CAUSAL_LM", bias="none",
)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    max_length=MAX_SEQ_LEN,
    logging_steps=10,
    eval_strategy="epoch",
    bf16=True,
    report_to=[],           # MLflow logging handled by autolog below
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
)

mlflow.set_experiment(MLFLOW_EXPERIMENT)
mlflow.transformers.autolog()

with mlflow.start_run(run_name="lora-trl") as run:
    mlflow.log_params({"base_model": BASE_MODEL, "lora_r": LORA_R,
                       "epochs": EPOCHS, "lr": LR})
    trainer.train()
    trainer.save_model(OUTPUT_DIR)   # saves the LoRA adapter
    print("Adapter saved to", OUTPUT_DIR)
    print("MLflow run:", run.info.run_id)

## (Optional) merge adapter into base weights

For serving without a PEFT runtime, merge the adapter and save full weights. Skip if you plan to serve the adapter directly.

In [ ]:
# from peft import PeftModel
# merged = PeftModel.from_pretrained(model, OUTPUT_DIR).merge_and_unload()
# merged.save_pretrained(f"{VOLUME}/merged_model")
# tokenizer.save_pretrained(f"{VOLUME}/merged_model")
print("Fine-tuned adapter is at:", OUTPUT_DIR, "→ evaluate in 03_evaluate.ipynb")